In [ ]:
# membandit setup
import os
import sys
from pathlib import Path

# Ensure cleaned_repo (parent of membandit/) is on sys.path when running from notebooks/
_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "membandit" / "__init__.py").exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

# Optional: set MEMORYBANDIT_DATA_DIR to your data root (defaults to /Volumes/T7 Shield)
# os.environ.setdefault("MEMORYBANDIT_DATA_DIR", "/path/to/data/root")

from membandit.plotting_style import set_plot_style
set_plot_style()


# MemoryBandit Pairwise Granger Causality (RPE)

Single-electrode HPC->ACC Granger causality, split by `rpe_cat`, using the same cue-aligned theta-band configuration as the MVGC RPE notebook.


In [ ]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

import os
import json
import hashlib
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from membandit.plotting_style import set_plot_style
from membandit.behavior import load_behavior_tables, preprocess_learn_df, preprocess_combined_df
from membandit.io import load_subject_epochs
from membandit.granger import enumerate_electrode_pairs, run_or_load_subject_pairwise_gc_rpe_df, load_pairwise_gc_rpe_df_cache, merge_with_behavior_rpe

warnings.filterwarnings('ignore')
set_plot_style()


In [ ]:
# Paths and behavior tables
base_dir = '/Volumes/T7 Shield'
load_dir = f'{base_dir}/work/qasims01/MemoryBanditData/EMU'
savepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/Connectivity/scratch/uvgc_rpe'
plot_savepath = '/Users/salmanqasim/Documents/Projects/MemBanditsEMU/Plots/MemoryBandit-UnivariateGrangerCausality_RPE'
os.makedirs(savepath, exist_ok=True)
os.makedirs(plot_savepath, exist_ok=True)

demo_file = f'{base_dir}/projects/guLab/Salman/EMU/MemoryBanditsDatabase.xlsx'
demo_data = pd.read_excel(demo_file, engine='openpyxl')
subj_df = demo_data[(demo_data.LongVersion == 1) & (demo_data.Minerva == 1)].copy()
subj_df = subj_df[~subj_df['MSSMCode'].isin(['MS015', 'MS024', 'MS034', 'MS038'])]
subjects = sorted(subj_df['MSSMCode'].dropna().unique().tolist())

learn_df_raw, combined_df_raw = load_behavior_tables(load_dir)
learn_df = preprocess_learn_df(learn_df_raw)
combined_df = preprocess_combined_df(combined_df_raw)

ROI_PAIRS = [('HPC', 'ACC')]

print('Subjects:', len(subjects))
print('learn_df:', learn_df.shape)
print('combined_df:', combined_df.shape)


In [ ]:
# Analysis configuration
ANALYSIS = {
    'event': 'cue_on',
    'day': 'Day1',
    'tmin': 0.0,
    'tmax': 1.0,
    'target_sfreq': 50,
    'fmin': 4,
    'fmax': 8,
    'gc_n_lags': 10,
    'n_freqs': None,
    'min_trials': 5,
}

USE_SURROGATES = True
N_SURROGATES_SMOKE = 100
N_SURROGATES_BATCH = 100
SURROGATE_TAIL = 'greater'
SURROGATE_RANDOM_STATE = 42
N_JOBS_SURROGATES_SMOKE = 12
N_JOBS_SURROGATES_BATCH = 12
SURROGATE_PARALLEL_BACKEND = 'loky'

PAIRWISE_GC_RPE_CACHE_DIR = os.path.join(savepath, 'pairwise_gc_df_cache_rpe')
OVERWRITE_PAIRWISE_GC_RPE_CACHE = True
RUN_SINGLE_SUBJECT_SMOKE = True

ANALYSIS_RESULTS = globals().get('ANALYSIS_RESULTS', {})

def _analysis_value_token(value):
    return str(value).replace('.', 'p').replace('-', 'm')

def _normalize_roi_pairs(pairs):
    return sorted([list(pair) for pair in pairs])

def _build_analysis_tag(analysis, roi_pairs):
    roi_pairs_norm = _normalize_roi_pairs(roi_pairs)
    roi_hash = hashlib.sha1(json.dumps(roi_pairs_norm, sort_keys=True).encode('utf-8')).hexdigest()[:10]
    band_tag_local = f"{analysis['fmin']}-{analysis['fmax']}Hz"
    return (
        f"pairwise_{analysis['event']}_{analysis['day']}"
        f"_t{_analysis_value_token(analysis['tmin'])}-{_analysis_value_token(analysis['tmax'])}"
        f"_sf{_analysis_value_token(analysis['target_sfreq'])}"
        f"_{band_tag_local}"
        f"_lag{_analysis_value_token(analysis['gc_n_lags'])}"
        f"_roi{roi_hash}"
    )

analysis_tag = _build_analysis_tag(ANALYSIS, ROI_PAIRS)
ANALYSIS_DIR = os.path.join(savepath, "notebook_runs", analysis_tag)
os.makedirs(ANALYSIS_DIR, exist_ok=True)

def get_analysis_paths(tag=None):
    tag = analysis_tag if tag is None else tag
    base_dir_local = os.path.join(savepath, 'notebook_runs', tag)
    os.makedirs(base_dir_local, exist_ok=True)
    return {
        'base_dir': base_dir_local,
        'all_pairs_csv': os.path.join(base_dir_local, 'all_pairs_pairwise_gc_rpe.csv'),
        'cache_log_csv': os.path.join(base_dir_local, 'cache_log_pairwise_gc_rpe.csv'),
        'merged_gc_csv': os.path.join(base_dir_local, 'merged_pairwise_gc_rpe.csv'),
        'smoke_df_csv': os.path.join(base_dir_local, 'smoke_pairwise_gc_rpe.csv'),
        'smoke_meta_json': os.path.join(base_dir_local, 'smoke_pairwise_gc_rpe_meta.json'),
        'summary_csv': os.path.join(base_dir_local, 'pairwise_gc_rpe_summary.csv'),
        'contrast_csv': os.path.join(base_dir_local, 'pairwise_gc_rpe_contrasts.csv'),
        'heatmap_pdf': os.path.join(base_dir_local, 'pairwise_gc_rpe_heatmaps.pdf'),
        'ranked_pdf': os.path.join(base_dir_local, 'pairwise_gc_rpe_ranked_pairs.pdf'),
    }

analysis_paths = get_analysis_paths()
print('Analysis tag:', analysis_tag)
print('Analysis dir:', ANALYSIS_DIR)


In [ ]:
# Find a smoke subject with at least one same-hemisphere HPC->ACC pair
def find_smoke_subject(subject_ids):
    rows = []
    for subj in subject_ids:
        epoch_data = load_subject_epochs(
            subj_id=subj,
            base_dir=base_dir,
            event=ANALYSIS['event'],
            roi_source='HPC',
            roi_target='ACC',
            tmin=ANALYSIS['tmin'],
            tmax=ANALYSIS['tmax'],
            target_sfreq=ANALYSIS['target_sfreq'],
            day=ANALYSIS['day'],
            fmin=ANALYSIS['fmin'],
            fmax=ANALYSIS['fmax'],
        )
        if epoch_data is None:
            continue
        pairs = enumerate_electrode_pairs(epoch_data, same_hemisphere_only=True)
        rows.append({'participant': subj, 'n_pairs': len(pairs)})
        if len(pairs) > 0:
            return subj, pd.DataFrame(rows)
    return None, pd.DataFrame(rows)

SMOKE_SUBJECT, pair_availability_df = find_smoke_subject(subjects)
if SMOKE_SUBJECT is None:
    raise RuntimeError('No subject with same-hemisphere HPC->ACC electrode pairs was found.')
print('Smoke subject:', SMOKE_SUBJECT)
display(pair_availability_df.head())


In [ ]:
# Smoke run + cache reload validation
smoke_df = pd.DataFrame()
smoke_result = None
smoke_meta = {}

if RUN_SINGLE_SUBJECT_SMOKE:
    smoke_df, smoke_result, smoke_meta = run_or_load_subject_pairwise_gc_rpe_df(
        subj_id=SMOKE_SUBJECT,
        roi_source='HPC',
        roi_target='ACC',
        base_dir=base_dir,
        learn_df=learn_df,
        cache_dir=PAIRWISE_GC_RPE_CACHE_DIR,
        overwrite_cache=OVERWRITE_PAIRWISE_GC_RPE_CACHE,
        return_result_dict=True,
        event=ANALYSIS['event'],
        tmin=ANALYSIS['tmin'],
        tmax=ANALYSIS['tmax'],
        target_sfreq=ANALYSIS['target_sfreq'],
        fmin=ANALYSIS['fmin'],
        fmax=ANALYSIS['fmax'],
        gc_n_lags=ANALYSIS['gc_n_lags'],
        n_freqs=ANALYSIS['n_freqs'],
        min_trials=ANALYSIS['min_trials'],
        day=ANALYSIS['day'],
        use_surrogates=USE_SURROGATES,
        n_surrogates=N_SURROGATES_SMOKE,
        surrogate_tail=SURROGATE_TAIL,
        random_state=SURROGATE_RANDOM_STATE,
        n_jobs_surrogates=N_JOBS_SURROGATES_SMOKE,
        surrogate_parallel_backend=SURROGATE_PARALLEL_BACKEND,
        store_surrogates=False,
        verbose=True,
        same_hemisphere_only=True,
    )
    smoke_df.to_csv(analysis_paths['smoke_df_csv'], index=False)
    with open(analysis_paths['smoke_meta_json'], "w") as f:
        json.dump(smoke_meta, f, indent=2, default=str)

required_cols = {'participant', 'hemisphere', 'source_electrode', 'target_electrode', 'pair_id', 'rpe_cat', 'time', 'gc_forward', 'gc_backward', 'gc_net'}
missing_cols = sorted(required_cols.difference(smoke_df.columns))
assert not missing_cols, f"Missing smoke columns: {missing_cols}"
assert smoke_df['pair_id'].notna().all()
pair_lookup = smoke_df[['participant', 'hemisphere', 'pair_id', 'source_electrode', 'target_electrode']].drop_duplicates()
assert pair_lookup.groupby(['participant', 'hemisphere', 'pair_id']).size().max() == 1

smoke_df_reload, _, smoke_meta_reload = run_or_load_subject_pairwise_gc_rpe_df(
    subj_id=SMOKE_SUBJECT,
    roi_source='HPC',
    roi_target='ACC',
    base_dir=base_dir,
    learn_df=learn_df,
    cache_dir=PAIRWISE_GC_RPE_CACHE_DIR,
    overwrite_cache=False,
    return_result_dict=False,
    event=ANALYSIS['event'],
    tmin=ANALYSIS['tmin'],
    tmax=ANALYSIS['tmax'],
    target_sfreq=ANALYSIS['target_sfreq'],
    fmin=ANALYSIS['fmin'],
    fmax=ANALYSIS['fmax'],
    gc_n_lags=ANALYSIS['gc_n_lags'],
    n_freqs=ANALYSIS['n_freqs'],
    min_trials=ANALYSIS['min_trials'],
    day=ANALYSIS['day'],
    use_surrogates=USE_SURROGATES,
    n_surrogates=N_SURROGATES_SMOKE,
    surrogate_tail=SURROGATE_TAIL,
    random_state=SURROGATE_RANDOM_STATE,
    n_jobs_surrogates=N_JOBS_SURROGATES_SMOKE,
    surrogate_parallel_backend=SURROGATE_PARALLEL_BACKEND,
    same_hemisphere_only=True,
)
assert smoke_meta_reload.get('cache_hit', False)
assert set(smoke_df.columns) == set(smoke_df_reload.columns)

print('Smoke cache file:', smoke_meta.get('cache_file'))
print('Smoke rows:', len(smoke_df))
print('Smoke pairs:', pair_lookup['pair_id'].nunique())
display(pair_lookup.head())
display(smoke_df.head())


In [ ]:
# Quick smoke visualization
if len(smoke_df) > 0:
    first_pair = smoke_df['pair_id'].dropna().iloc[0]
    plot_df = smoke_df[smoke_df['pair_id'].eq(first_pair)].copy()
    y_col = "gc_net_z" if plot_df['gc_net_z'].notna().any() else "gc_net"
    plt.figure(figsize=(8, 4))
    sns.lineplot(data=plot_df, x='time', y=y_col, hue='rpe_cat', errorbar=None)
    plt.axhline(0, color='gray', linewidth=1)
    plt.title(f"Smoke pair: {first_pair}")
    plt.tight_layout()
    plt.show()


In [ ]:
# Full batch run across subjects
analysis_state = ANALYSIS_RESULTS.setdefault(analysis_tag, {})
if "all_pairwise_gc_df" in analysis_state:
    all_pairwise_gc_df = analysis_state['all_pairwise_gc_df'].copy()
    cache_log_df = analysis_state.get('cache_log_df', pd.DataFrame()).copy()
elif (not OVERWRITE_PAIRWISE_GC_RPE_CACHE) and os.path.exists(analysis_paths['all_pairs_csv']):
    all_pairwise_gc_df = pd.read_csv(analysis_paths['all_pairs_csv'])
    cache_log_df = pd.read_csv(analysis_paths['cache_log_csv']) if os.path.exists(analysis_paths['cache_log_csv']) else pd.DataFrame()
else:
    all_rows = []
    cache_log_rows = []
    for subj in subjects:
        subj_df, _, meta = run_or_load_subject_pairwise_gc_rpe_df(
            subj_id=subj,
            roi_source='HPC',
            roi_target='ACC',
            base_dir=base_dir,
            learn_df=learn_df,
            cache_dir=PAIRWISE_GC_RPE_CACHE_DIR,
            overwrite_cache=OVERWRITE_PAIRWISE_GC_RPE_CACHE,
            return_result_dict=False,
            event=ANALYSIS['event'],
            tmin=ANALYSIS['tmin'],
            tmax=ANALYSIS['tmax'],
            target_sfreq=ANALYSIS['target_sfreq'],
            fmin=ANALYSIS['fmin'],
            fmax=ANALYSIS['fmax'],
            gc_n_lags=ANALYSIS['gc_n_lags'],
            n_freqs=ANALYSIS['n_freqs'],
            min_trials=ANALYSIS['min_trials'],
            day=ANALYSIS['day'],
            use_surrogates=USE_SURROGATES,
            n_surrogates=N_SURROGATES_BATCH,
            surrogate_tail=SURROGATE_TAIL,
            random_state=SURROGATE_RANDOM_STATE,
            n_jobs_surrogates=N_JOBS_SURROGATES_BATCH,
            surrogate_parallel_backend=SURROGATE_PARALLEL_BACKEND,
            store_surrogates=False,
            verbose=False,
            same_hemisphere_only=True,
        )
        cache_log_rows.append({
            'participant': subj,
            'cache_hit': bool(meta.get('cache_hit', False)),
            'cache_file': meta.get('cache_file', ''),
            'loaded_rows': int(len(subj_df)),
        })
        if len(subj_df) > 0:
            all_rows.append(subj_df)
    all_pairwise_gc_df = pd.concat(all_rows, ignore_index=True) if len(all_rows) > 0 else pd.DataFrame()
    cache_log_df = pd.DataFrame(cache_log_rows)
    all_pairwise_gc_df.to_csv(analysis_paths['all_pairs_csv'], index=False)
    cache_log_df.to_csv(analysis_paths['cache_log_csv'], index=False)

analysis_state['all_pairwise_gc_df'] = all_pairwise_gc_df.copy()
analysis_state['cache_log_df'] = cache_log_df.copy()
print('Total exported rows:', len(all_pairwise_gc_df))
display(cache_log_df.head())
display(all_pairwise_gc_df.head())


In [ ]:
# Behavior-merged export
if len(all_pairwise_gc_df) == 0:
    merged_pairwise_gc_df = pd.DataFrame()
else:
    merged_pairwise_gc_df = merge_with_behavior_rpe(all_pairwise_gc_df.copy(), learn_df, event=ANALYSIS['event'])
    merged_pairwise_gc_df.to_csv(analysis_paths['merged_gc_csv'], index=False)

ANALYSIS_RESULTS.setdefault(analysis_tag, {})['merged_pairwise_gc_df'] = merged_pairwise_gc_df.copy()
print('Merged rows:', len(merged_pairwise_gc_df))
if len(merged_pairwise_gc_df) > 0:
    assert "rpe_cat" in merged_pairwise_gc_df.columns
display(merged_pairwise_gc_df.head())


In [ ]:
# Pair summary + contrasts
summary_source_df = merged_pairwise_gc_df.copy() if len(merged_pairwise_gc_df) > 0 else all_pairwise_gc_df.copy()
summary_df = pd.DataFrame()
contrast_df = pd.DataFrame()

if len(summary_source_df) > 0:
    metric_col = "gc_net_z" if summary_source_df['gc_net_z'].notna().any() else "gc_net"
    group_cols = ['participant', 'hemisphere', 'source_electrode', 'target_electrode', 'pair_id', 'rpe_cat']
    summary_df = (
        summary_source_df.groupby(group_cols, dropna=False)
        .agg(
            mean_gc=(metric_col, "mean"),
            mean_gc_raw=('gc_net', "mean"),
            n_timepoints=('time', "size"),
            n_trials=('n_trials', "max"),
        )
        .reset_index()
    )
    summary_df.to_csv(analysis_paths['summary_csv'], index=False)

    contrast_df = summary_df.pivot_table(
        index=['participant', 'hemisphere', 'source_electrode', 'target_electrode', 'pair_id'],
        columns='rpe_cat',
        values='mean_gc',
        aggfunc='mean',
    ).reset_index()
    contrast_df.columns.name = None
    for cond in ['neg', 'neutral', 'pos']:
        if cond not in contrast_df.columns:
            contrast_df[cond] = np.nan
    contrast_df['pos_minus_neutral'] = contrast_df['pos'] - contrast_df['neutral']
    contrast_df['pos_minus_neg'] = contrast_df['pos'] - contrast_df['neg']
    contrast_df['neutral_minus_neg'] = contrast_df['neutral'] - contrast_df['neg']
    contrast_df.to_csv(analysis_paths['contrast_csv'], index=False)

    assert contrast_df[['participant', 'pair_id']].drop_duplicates().shape[0] == len(contrast_df)

display(summary_df.head())
display(contrast_df.head())


In [ ]:
# Pairwise heatmaps by hemisphere
if len(contrast_df) > 0:
    heatmap_df = (
        contrast_df.groupby(['hemisphere', 'source_electrode', 'target_electrode'], dropna=False)['pos_minus_neutral']
        .mean()
        .reset_index()
    )
    hemi_order = [hemi for hemi in ['l', 'r'] if hemi in heatmap_df['hemisphere'].unique()]
    if len(hemi_order) == 0:
        hemi_order = sorted(heatmap_df['hemisphere'].dropna().unique().tolist())
    fig, axes = plt.subplots(1, max(1, len(hemi_order)), figsize=(7 * max(1, len(hemi_order)), 6), squeeze=False)
    axes = axes.ravel()
    for ax, hemi in zip(axes, hemi_order):
        hemi_df = heatmap_df[heatmap_df['hemisphere'].eq(hemi)].copy()
        pivot_df = hemi_df.pivot(index='source_electrode', columns='target_electrode', values='pos_minus_neutral')
        sns.heatmap(pivot_df, cmap='coolwarm', center=0, annot=True, fmt='.2f', ax=ax)
        ax.set_title(f"{hemi.upper()} hemisphere | pos - neutral")
        ax.set_xlabel('ACC electrode')
        ax.set_ylabel('HPC electrode')
    for ax in axes[len(hemi_order):]:
        ax.axis('off')
    plt.tight_layout()
    fig.savefig(analysis_paths['heatmap_pdf'], dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()


In [ ]:
# Ranked pairs + strip plot
if len(summary_df) > 0 and len(contrast_df) > 0:
    ranked_pairs = contrast_df.sort_values('pos_minus_neutral', ascending=False).head(12).copy()
    display(ranked_pairs[['participant', 'hemisphere', 'source_electrode', 'target_electrode', 'pair_id', 'pos_minus_neutral', 'pos_minus_neg', 'neutral_minus_neg']])

    plot_pairs = ranked_pairs['pair_id'].dropna().unique().tolist()[:6]
    strip_df = summary_df[summary_df['pair_id'].isin(plot_pairs) & summary_df['rpe_cat'].isin(['neutral', 'pos'])].copy()
    if len(strip_df) > 0:
        plt.figure(figsize=(max(8, len(plot_pairs) * 1.4), 5))
        sns.stripplot(
            data=strip_df,
            x='pair_id',
            y='mean_gc',
            hue='rpe_cat',
            dodge=True,
            palette={'neutral': "#7f7f7f", 'pos': "#2ca02c"},
            alpha=0.75,
            size=5,
        )
        plt.axhline(0, color='gray', linewidth=1)
        plt.xticks(rotation=45, ha='right')
        plt.ylabel('Mean gc_net_z' if summary_df['mean_gc'].notna().any() else "Mean gc_net")
        plt.title('Top HPC->ACC pairs driving pos vs neutral')
        plt.tight_layout()
        plt.savefig(analysis_paths['ranked_pdf'], dpi=300, bbox_inches='tight', facecolor='white')
        plt.show()

print('Summary CSV:', analysis_paths['summary_csv'])
print('Contrast CSV:', analysis_paths['contrast_csv'])
print('Heatmap PDF:', analysis_paths['heatmap_pdf'])
print('Ranked PDF:', analysis_paths['ranked_pdf'])
